**Code :** [GitHub · Kukai-7/industrial-data-valuation](https://github.com/Kukai-7/industrial-data-valuation) · **Données synthétiques :** [Hugging Face · synthetic-industrial-data-valuation](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · **Démo :** [Space](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)

**Résumé (FR).** Comment classer la valeur d'un portefeuille de données industrielles traité comme actif incorporel ? Huit jeux anonymisés sont notés sur 17 critères de conformité, puis valorisés par trois approches (coût, marché, revenus) fondues selon leur confiance, avec sensibilité, Monte Carlo et pondération AHP. Limites d'abord : les montants sont des hypothèses de travail et des valeurs relatives, pas un prix opposable.

**Abstract (EN).** How can the value of an industrial data portfolio, treated as an intangible asset, be ranked? Eight anonymised datasets are scored on 17 compliance criteria, then valued with three approaches (cost, market, income) blended by confidence, with sensitivity, Monte Carlo and AHP weighting. Limits first: amounts are working assumptions giving a relative ranking, not an opposable price.

<div style="padding:28px 0 6px 0">
<div style="font-size:12px;letter-spacing:.14em;text-transform:uppercase;color:#898781">Document de recherche · octobre 2026</div>
<h1 style="font-size:34px;line-height:1.1;margin:10px 0 6px 0">Valoriser un portefeuille de données industrielles</h1>
<div style="font-size:17px;color:#6E6E73">Référentiel de conformité, valorisation hybride, exposition réglementaire, incertitude et ancrage académique</div>
</div>

Ce notebook rassemble **toute l'étude** dans un seul fichier exécutable : les hypothèses, le modèle, les résultats, leurs limites et les sources. Chaque graphique est recalculé à partir des cellules de paramètres, donc toute modification se propage d'un bout à l'autre.

**Comment le lire.** Les cellules grises contiennent le code, les autres le raisonnement. Pour tout recalculer : *Run All* (ou `Ctrl + Maj + P` puis « Run All Cells » dans VS Code). Paquets nécessaires : `numpy`, `pandas`, `matplotlib` (voir `requirements.txt`).

| | Partie | Ce qu'elle établit |
|---|---|---|
| 1 | Données et moteur | Les 17 critères, les 8 jeux, les paramètres, les formules |
| 2 | Synthèse | Les quatre chiffres qui résument l'étude |
| 3 | Cadre | Le parcours en trois étapes et la place de la grille |
| 4 | Conformité | Qui franchit les seuils, et pourquoi |
| 5 | Valorisation | Trois approches fondues par la confiance |
| 6 | Correction | La conformité trie la valeur sans la gonfler |
| 7 | Attribution | Ce qui revient réellement au groupe |
| 8 | Exposition et priorités | Où agir d'abord |
| 9 | Gains par critère | Quel point de conformité rapporte le plus |
| 10 | Sensibilité | Les hypothèses qui font bouger le résultat |
| 11 | Monte Carlo | La fourchette plutôt que le point |
| 12 | Pondération AHP | Dériver les poids au lieu de les poser |
| 13 | Ancrage externe | Comparables de marché et écart perçu / reconnu |
| 14 | Flux futurs | La valeur des données d'un parc installé |
| 15 | Veille | Le calendrier réglementaire jusqu'en octobre 2026 |
| 16 | Littérature | Convergence avec l'état de l'art |
| 17 | Limites | Ce que l'étude ne prouve pas |

> **Règle de lecture.** Toutes les valeurs sont **relatives** : elles servent à classer les jeux entre eux, pas à fixer un prix. Une valeur n'est pas un prix : un prix suppose un benchmark externe.

## Mise en place

Une seule palette pour tout le document : un bleu pour la magnitude, trois teintes fixes pour les trois approches de valeur, et une palette de statut (vert, ambre, rouge) réservée aux verdicts de conformité, toujours doublée d'un libellé.

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ── Palette (validée daltonisme, mode clair) ─────────────────────────────
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE, CARD = "#e1e0d9", "#c3c2b7", "#fcfcfb", "#f3f2ee"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"            # coût, marché, revenus
APPROCHES = {"Coût": BLUE, "Marché": ORANGE, "Revenus": AQUA}
GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"                # statut
RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
UP, DOWN = "#256abf", "#d03b3b"                                    # divergent
PRIO = {"P1": "#0d366b", "P2": "#256abf", "P3": "#5598e7", "P4": "#86b6ef"}

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 10.5, "axes.titlesize": 14, "axes.titleweight": "semibold",
    "axes.titlelocation": "left", "axes.titlepad": 26, "axes.labelcolor": INK2,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "legend.frameon": False, "axes.axisbelow": True,
})

def titre(ax, t, sous=None):
    # Titre à gauche + sous-titre gris (une phrase qui dit quoi lire).
    ax.set_title(t, loc="left", color=INK)
    if sous:
        ax.text(0, 1.02, sous, transform=ax.transAxes, color=INK2, fontsize=10, va="bottom")

def titre_fig(fig, ax, t, sous=None, top=.84):
    # Titre posé au niveau de la figure (pour les cartes de chaleur à axe x en haut).
    fig.subplots_adjust(top=top)
    x0 = ax.get_position().x0
    fig.text(x0, .985, t, fontsize=14, fontweight="semibold", color=INK, va="top")
    if sous: fig.text(x0, .945, sous, fontsize=10, color=INK2, va="top")

def eur(v):
    a = abs(v); s = "−" if v < 0 else ""
    if a >= 1e6: return f"{s}{a/1e6:,.2f} M€".replace(",", " ").replace(".", ",")
    if a >= 1e3: return f"{s}{a/1e3:,.0f} k€".replace(",", " ")
    return f"{s}{a:,.0f} €".replace(",", " ")

def pct(v, d=1):
    return f"{v*100:+.{d}f} %".replace(".", ",").replace("-", "−")

def tableau(df, fmt=None, barre=None):
    # Tableau lisible en thème clair comme sombre.
    sty = df.style.hide(axis="index")
    if fmt: sty = sty.format(fmt)
    sty = sty.set_table_styles([
        {"selector": "th", "props": "text-align:left;font-weight:600;padding:6px 12px;border-bottom:1px solid #c3c2b7"},
        {"selector": "td", "props": "padding:5px 12px;border-bottom:1px solid #e1e0d9"},
        {"selector": "", "props": "border-collapse:collapse;font-size:12.5px"},
    ])
    if barre: sty = sty.bar(subset=barre, color="#9ec5f4", vmin=0)
    return sty

print("Prêt.")

## 1 · Données d'entrée et moteur

### 1.1 Le référentiel en 17 critères

Les critères reprennent la structure des lignes directrices nationales sur la construction de jeux de données de haute qualité (août 2025) : **documentation** (4), **qualité intrinsèque** (8), **application au modèle** (5). Chaque critère est noté de 1 à 5. Seul le dernier critère, l'adéquation au modèle, se prouve par un test plutôt que par une déclaration, d'où le poids élevé du bloc C.

In [ ]:
CRITERES = pd.DataFrame([
    ("A1","Informations de base","A","Volumétrie, format, structure, canal d'accès"),
    ("A2","Caractéristiques de contenu","A","Modalités, distribution, classes, limites"),
    ("A3","Processus de construction","A","Sources, collecte, traitement, versions"),
    ("A4","Notice d'application","A","Licence, cas d'usage, résultats de référence"),
    ("B1","Conformité de format","B","Format standard, directement exploitable"),
    ("B2","Conformité de sécurité","B","Aucun contenu illicite ni atteinte aux droits"),
    ("B3","Conformité d'annotation","B","Règle écrite, appliquée de bout en bout"),
    ("B4","Complétude structurelle","B","Métadonnées complètes, manquants maîtrisés"),
    ("B5","Véracité du contenu","B","Traçable jusqu'à la source"),
    ("B6","Cohérence du contenu","B","Concordance entre données liées"),
    ("B7","Cohérence de type","B","Conforme au niveau de spécialisation revendiqué"),
    ("B8","Propreté du contenu","B","Ni doublons, ni bruit"),
    ("C1","Diversité du contenu","C","Couverture du cas d'usage, sans angle mort"),
    ("C2","Complétude d'échelle","C","Volume suffisant pour le modèle cible"),
    ("C3","Fraîcheur","C","Collecte et mise à jour compatibles"),
    ("C4","Exactitude de l'annotation","C","L'annotation marque toute l'information utile"),
    ("C5","Adéquation au modèle","C","Gain de performance mesuré par un test"),
], columns=["code","critère","bloc","ce qu'il vérifie"])
BLOCS = {"A": "Documentation", "B": "Qualité intrinsèque", "C": "Application au modèle"}
tableau(CRITERES)

### 1.2 Les huit jeux de données

Le portefeuille anonymisé d'un **grand site industriel**, réparti entre une filiale détenue à 100 % et une coentreprise détenue à 50 %. Les montants sont des hypothèses de travail cohérentes entre elles, pas des données comptables.

In [ ]:
JEUX = pd.DataFrame([
 # nom, court, entité, part, ETP-mois, coût ETP, infra, volume, prix réf., décote, conf. marché, bénéfice/an, économie/an, horizon, facteur v1
 ("Télémétrie capteurs production","Télémétrie","Filiale 100 %",1.0,18,9000,120000,500000,0.8,0.40,0.55,150000,400000,5,1.00),
 ("Historique maintenance","Maintenance","Coentreprise",0.5,30,9000,200000,1200000,1.1,0.45,0.50,300000,600000,5,1.10),
 ("Qualité et défauts fournisseurs","Fournisseurs","Filiale 100 %",1.0,10,8500,60000,300000,0.6,0.50,0.45,80000,250000,5,0.90),
 ("Logs énergétiques usine","Énergie","Filiale 100 %",1.0,8,8000,40000,800000,0.2,0.55,0.50,40000,180000,5,0.85),
 ("Chaîne d'approvisionnement","Appro","Coentreprise",0.5,14,8800,90000,600000,0.9,0.45,0.55,200000,350000,5,1.00),
 ("Paramètres procédés","Procédés","Coentreprise",0.5,25,9500,110000,150000,2.5,0.50,0.40,250000,700000,7,1.15),
 ("Données de personnel","Personnel","Filiale 100 %",1.0,6,8000,30000,50000,1.5,0.60,0.35,20000,90000,5,0.80),
 ("Essais et recherche","Essais","Coentreprise",0.5,40,10000,250000,200000,3.0,0.55,0.40,400000,900000,7,1.20),
], columns=["nom","court","entité","part","etp","cetp","infra","vol","prix","dec","confm","ben","eco","hor","fq1"])

# Notes des 17 critères (lignes = critères A1…C5, colonnes = jeux)
NOTES = pd.DataFrame({
 "Télémétrie":   [2,2,1,1,4,4,1,3,4,3,3,2,3,5,5,1,2],
 "Maintenance":  [4,4,3,4,4,4,4,4,5,4,5,4,4,4,4,4,4],
 "Fournisseurs": [2,2,2,2,3,3,2,2,3,2,3,3,3,3,3,2,3],
 "Énergie":      [2,2,1,1,4,5,1,3,4,3,1,3,1,4,5,1,1],
 "Appro":        [3,3,2,3,3,3,3,3,3,3,3,3,3,4,4,3,3],
 "Procédés":     [4,4,4,3,5,4,4,4,5,4,5,4,3,2,4,4,4],
 "Personnel":    [2,2,2,1,3,1,1,2,3,2,2,2,1,1,3,1,1],
 "Essais":       [4,4,4,4,4,3,4,4,5,4,5,4,5,3,4,4,5],
}, index=CRITERES.code)

# Exposition réglementaire : 5 critères notés de 1 à 5
EXPO_POIDS = {"Maturité de l'industrie locale": .30, "Criticité opérationnelle": .20,
              "Données personnelles": .20, "Sécurité nationale": .20, "Partage en coentreprise": .10}
EXPO = pd.DataFrame([[3,4,1,2,2],[5,4,2,3,4],[2,3,1,1,2],[3,2,1,3,1],
                     [3,3,2,2,4],[4,5,1,4,5],[2,2,5,2,1],[4,5,2,5,5]],
                    index=JEUX.court, columns=list(EXPO_POIDS))

vue = JEUX[["nom","entité","part","etp","infra","vol","prix","ben","eco","hor"]].copy()
vue.columns = ["Jeu de données","Détention","Part","ETP-mois","Infra (€)","Volume","Prix réf. (€)","Bénéfice/an (€)","Économie/an (€)","Horizon (ans)"]
tableau(vue, {"Part":"{:.0%}","Infra (€)":"{:,.0f}","Volume":"{:,.0f}","Bénéfice/an (€)":"{:,.0f}","Économie/an (€)":"{:,.0f}","Prix réf. (€)":"{:.1f}"})

### 1.3 Les paramètres du modèle

Tous réglables ici. Les trois poids d'approche (30 / 25 / 45) sont posés au jugement ; la partie 12 montre comment les dériver proprement.

In [ ]:
P = dict(
    wDoc=.20, wQual=.45, wApp=.35,          # poids des trois blocs du référentiel
    fMin=.70, fMax=1.30,                     # bornes du facteur qualité (approche coût)
    seuilConforme=75, seuilReserve=55,       # verdicts
    alpha=.30, beta=.50,                     # effet du score sur la décote et sur les revenus
    decMin=.05, decMax=.80,
    taux=.12,                                # actualisation des revenus
    pc=.30, pm=.25, pr=.45,                  # poids des approches coût / marché / revenus
    cc=.90, cr=.70,                          # confiance coût / revenus (marché : par jeu)
    seuilValeur=50, seuilExpo=50,            # priorisation
)
pd.Series(P).to_frame("valeur").T

### 1.4 Le moteur de calcul

Tout le modèle tient dans les fonctions ci-dessous. Le score de conformité agit sur les **trois** approches à la fois : il module le facteur qualité du coût, réduit la décote de marché et relève le multiplicateur de revenus.

$$V_{\text{hybride}} = \frac{p_c\,c_c\,V_c \;+\; p_m\,c_m\,V_m \;+\; p_r\,c_r\,V_r}{p_c\,c_c + p_m\,c_m + p_r\,c_r}$$

In [ ]:
def borne(x, a, b): return np.minimum(b, np.maximum(a, x))

def score_bloc(notes, bloc):
    idx = CRITERES.index[CRITERES.bloc == bloc]
    return (notes.iloc[idx].mean() - 1) / 4 * 100

def score_global(notes, P=P):
    return P["wDoc"]*score_bloc(notes,"A") + P["wQual"]*score_bloc(notes,"B") + P["wApp"]*score_bloc(notes,"C")

def annuite(h, t): return (1 - (1 + t) ** (-h)) / t

def valoriser(j, s, P=P, mode="apres"):
    # Renvoie les trois approches et la valeur hybride d'un jeu j pour un score s.
    if mode == "v1":
        f, dec, mul = j.fq1, j.dec, 1.0
    else:
        f   = borne(P["fMin"] + (s - 20) / 80 * (P["fMax"] - P["fMin"]), P["fMin"], P["fMax"])
        dec = borne(j.dec - P["alpha"] * (s - 50) / 100, P["decMin"], P["decMax"])
        mul = 1 + P["beta"] * (s - 50) / 100
    vc = (j.etp * j.cetp + j.infra) * f
    vm = j.vol * j.prix * (1 - dec)
    vr = (j.ben + j.eco) * mul * annuite(j.hor, P["taux"])
    wc, wm, wr = P["pc"]*P["cc"], P["pm"]*j.confm, P["pr"]*P["cr"]
    den = wc + wm + wr
    return dict(vc=vc, vm=vm, vr=vr, val=(wc*vc + wm*vm + wr*vr)/den,
                part_c=wc*vc, part_m=wm*vm, part_r=wr*vr, den=den, f=f, dec=dec, mul=mul)

def verdict(s, P=P):
    return "Conforme" if s >= P["seuilConforme"] else ("Sous réserve" if s >= P["seuilReserve"] else "Non conforme")

def calculer(P=P, J=JEUX, N=NOTES):
    lignes = []
    for _, j in J.iterrows():
        s = score_global(N[j.court], P)
        av, ap = valoriser(j, s, P, "v1"), valoriser(j, s, P, "apres")
        lignes.append(dict(court=j.court, nom=j.nom, entité=j.entité, part=j.part, score=s,
                           A=score_bloc(N[j.court],"A"), B=score_bloc(N[j.court],"B"), C=score_bloc(N[j.court],"C"),
                           verdict=verdict(s, P), avant=av["val"], apres=ap["val"],
                           vc=ap["vc"], vm=ap["vm"], vr=ap["vr"],
                           part_c=ap["part_c"]/(ap["val"]*ap["den"]), part_m=ap["part_m"]/(ap["val"]*ap["den"]), part_r=ap["part_r"]/(ap["val"]*ap["den"]),
                           facteur=ap["f"], attrib=ap["val"]*j.part, attrib_avant=av["val"]*j.part))
    R = pd.DataFrame(lignes).set_index("court")
    R["ecart"] = R.apres / R.avant - 1
    return R

R = calculer()
VERDICT_COUL = {"Conforme": GOOD, "Sous réserve": WARN, "Non conforme": CRIT}
TOT_AV, TOT_AP = R.attrib_avant.sum(), R.attrib.sum()
print(f"Portefeuille attribuable : {eur(TOT_AV)} → {eur(TOT_AP)}  ({pct(TOT_AP/TOT_AV-1)})")

## 2 · Synthèse

Les quatre chiffres à retenir, recalculés à chaque exécution.

In [ ]:
fig = plt.figure(figsize=(12, 2.9))
tuiles = [
    (f"{len(R)}", "jeux de données évalués", "sur 17 critères chacun", INK),
    (f"{(R.verdict=='Conforme').sum()} / {len(R)}", "jeux conformes", "les seuls présentables sans réserve", GOOD),
    (eur(TOT_AP), "valeur attribuable", f"portefeuille {pct(TOT_AP/TOT_AV-1)} après correction", BLUE),
    (f"{pct(R.ecart.min(),0)} à {pct(R.ecart.max(),0)}", "écart de correction par jeu", "la méthode trie, elle ne gonfle pas", INK),
]
for i, (v, l, s, c) in enumerate(tuiles):
    ax = fig.add_axes([0.005 + i*0.25, 0.04, 0.235, 0.9]); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.06",
                                transform=ax.transAxes, fc=CARD, ec="none"))
    ax.text(.08, .66, v, fontsize=25, fontweight="semibold", color=c, transform=ax.transAxes)
    ax.text(.08, .36, l, fontsize=11.5, color=INK, transform=ax.transAxes)
    ax.text(.08, .17, s, fontsize=9.5, color=MUTED, transform=ax.transAxes)
plt.show()

**En une phrase :** sur huit jeux, deux seulement franchissent le seuil de conformité ; la correction par la qualité déplace chaque jeu de −16 % à +12 % mais le portefeuille de +4,5 % seulement. La méthode ne fabrique pas de valeur, elle la redistribue vers les jeux qui la méritent.

## 3 · Le cadre : un parcours en trois étapes

Le directeur de l'Académie nationale de recherche sur le développement des données décrit un parcours de valorisation en trois temps : **ressourcisation** (inventaire, gouvernance), **productisation** (jeux livrables, API, modèles-services), **actifisation** (droits, bilan, circulation). La grille couvre la première et la troisième étape. La deuxième reste à construire : c'est la suite naturelle du travail.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 4)
etapes = [("1 · Ressourcisation", "Inventorier, gouverner,\nrendre traçable", "Procès-verbal en 17 critères", BLUE, False),
          ("2 · Productisation", "Jeux livrables, API,\nmodèles-services", "À construire", MUTED, True),
          ("3 · Actifisation", "Droits, mise au bilan,\ncirculation", "Grille de valorisation", BLUE, False)]
for i, (t, d, outil, c, vide) in enumerate(etapes):
    x = 0.2 + i * 4
    ax.add_patch(FancyBboxPatch((x, 1.9), 3.5, 1.8, boxstyle="round,pad=0.02,rounding_size=0.18",
                                fc=SURFACE if vide else CARD, ec=c, lw=1.4, ls="--" if vide else "-"))
    ax.text(x + .25, 3.25, t, fontsize=13, fontweight="semibold", color=INK)
    ax.text(x + .25, 2.15, d, fontsize=10, color=INK2, va="bottom")
    ax.add_patch(FancyBboxPatch((x, .3), 3.5, 1.1, boxstyle="round,pad=0.02,rounding_size=0.18",
                                fc=SURFACE, ec=c, lw=1.0, ls=":" if vide else "-"))
    ax.text(x + 1.75, .85, outil, fontsize=10.5, ha="center", va="center",
            color=WARN if vide else INK, fontweight="semibold" if vide else "normal")
    ax.annotate("", (x + 1.75, 1.42), (x + 1.75, 1.88), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1))
    if i < 2:
        ax.annotate("", (x + 3.95, 2.8), (x + 3.55, 2.8), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1.2))
ax.text(0.2, 3.95, "Parcours national de valorisation", fontsize=14, fontweight="semibold")
plt.show()

> **Nuance indispensable.** Un score élevé prouve que la donnée est *prête* pour l'actifisation, pas qu'elle est devenue un actif. L'actifisation réelle suppose deux actes que la grille ne mesure pas : l'enregistrement des droits de propriété et la mise au bilan. C'est d'ailleurs la phase qu'une bourse de données nationale qualifie de « plus critique » dans le premier cas de boucle fermée observé (août 2026).

## 4 · Conformité : qui franchit les seuils

### 4.1 La carte des notes

Chaque case est une note de 1 à 5. Les colonnes sont triées par score global. Le contraste se lit d'un coup d'œil : les jeux les mieux notés le sont partout, les plus faibles décrochent surtout sur la documentation et l'annotation.

In [ ]:
ordre = R.score.sort_values(ascending=False).index
M = NOTES[ordre]
cmap = mpl.colors.ListedColormap(["#e9f1fb", "#b7d3f6", "#6da7ec", "#2a78d6", "#104281"])
fig, ax = plt.subplots(figsize=(11.5, 7.2))
ax.imshow(M.values, cmap=cmap, vmin=.5, vmax=5.5, aspect="auto")
for (i, k), v in np.ndenumerate(M.values):
    ax.text(k, i, v, ha="center", va="center", fontsize=10, color="white" if v >= 4 else INK)
ax.set_xticks(range(len(ordre)), [f"{c}\n{R.score[c]:.0f}/100" for c in ordre], color=INK, fontsize=10)
ax.set_yticks(range(17), [f"{c}  {n}" for c, n in zip(CRITERES.code, CRITERES["critère"])], color=INK2, fontsize=9.5)
ax.xaxis.tick_top(); ax.grid(False)
for y in (3.5, 11.5): ax.axhline(y, color=SURFACE, lw=4)
for sp in ax.spines.values(): sp.set_visible(False)
ax.tick_params(length=0)
titre_fig(fig, ax, "Les 17 notes de chaque jeu, du mieux au moins bien noté",
          "Note de 1 (absent) à 5 (conforme et vérifié) · les bandes séparent documentation, qualité, application")
plt.show()

### 4.2 Le score global face aux deux seuils

Le score est la moyenne pondérée des trois blocs (20 % / 45 % / 35 %), ramenée sur 100. Seuil « conforme » à 75, seuil « sous réserve » à 55.

In [ ]:
r = R.sort_values("score")
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.axvspan(0, P["seuilReserve"], color=CRIT, alpha=.05, lw=0)
ax.axvspan(P["seuilReserve"], P["seuilConforme"], color=WARN, alpha=.08, lw=0)
ax.axvspan(P["seuilConforme"], 100, color=GOOD, alpha=.06, lw=0)
for i, (c, row) in enumerate(r.iterrows()):
    col = VERDICT_COUL[row.verdict]
    ax.plot([0, row.score], [i, i], color=GRID, lw=1.2, zorder=1)
    for b, m in (("A", "^"), ("B", "s"), ("C", "D")):
        ax.scatter(row[b], i, marker=m, s=26, color="white", edgecolor=MUTED, lw=.9, zorder=2)
    ax.scatter(row.score, i, s=110, color=col, edgecolor="white", lw=2, zorder=3)
    ax.text(102, i, f"{row.score:.0f}", va="center", fontsize=10.5, fontweight="semibold", color=INK, clip_on=False)
    ax.text(108, i, row.verdict, va="center", fontsize=9.5, color=col if row.verdict != "Sous réserve" else "#9a6a00", clip_on=False)
ax.set_yticks(range(len(r)), r.index, color=INK)
ax.set_xlim(0, 100); ax.set_ylim(-1.1, len(r) - .5); ax.grid(axis="y", visible=False)
for s, l in ((P["seuilReserve"], "seuil 55"), (P["seuilConforme"], "seuil 75")):
    ax.axvline(s, color=AXIS, ls="--", lw=1)
    ax.text(s, -.85, l, ha="center", fontsize=9, color=MUTED)
fig.subplots_adjust(right=.8)
ax.scatter([], [], marker="^", color="white", edgecolor=MUTED, label="Documentation")
ax.scatter([], [], marker="s", color="white", edgecolor=MUTED, label="Qualité")
ax.scatter([], [], marker="D", color="white", edgecolor=MUTED, label="Application")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.1), ncol=3, fontsize=9)
titre(ax, "Deux jeux sur huit franchissent le seuil de conformité",
      "Gros point : score global · petits symboles : score de chaque bloc (sur 100)")
plt.show()

## 5 · Valorisation : trois approches, une valeur

### 5.1 Les trois montants par jeu

Les trois approches ne racontent pas la même histoire. L'approche par les revenus domine presque partout ; l'approche marché reste la plus fragile faute de comparables, d'où sa confiance la plus faible.

In [ ]:
r = R.sort_values("apres")
fig, ax = plt.subplots(figsize=(11, 5.6))
h, y = .24, np.arange(len(r))
for k, (lab, col) in enumerate(APPROCHES.items()):
    vals = r[["vc", "vm", "vr"][k]] / 1e6
    ax.barh(y + (1 - k) * h, vals, height=h - .03, color=col, label=lab)
ax.scatter(r.apres / 1e6, y, marker="|", s=420, color=INK, lw=2, zorder=4, label="Valeur hybride retenue")
for i, v in enumerate(r.apres): ax.text(v/1e6, i + .44, eur(v), ha="center", fontsize=8.5, color=INK)
ax.set_yticks(y, r.index, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel("millions d'euros")
ax.legend(loc="lower right", fontsize=9.5)
titre(ax, "Les revenus pèsent plus lourd que le coût et le marché réunis",
      "Montant de chaque approche, et valeur hybride retenue (trait noir), avant attribution")
plt.show()

### 5.2 D'où vient la valeur retenue

Contribution de chaque approche à la valeur hybride, une fois pondérée par la confiance. C'est la **faiblesse assumée** du modèle : la jambe revenus pèse le plus, alors que c'est la plus dépendante d'hypothèses de bénéfices futurs.

In [ ]:
r = R.sort_values("part_r")
fig, ax = plt.subplots(figsize=(11, 4.8))
gauche = np.zeros(len(r))
for k, (lab, col) in enumerate(APPROCHES.items()):
    v = r[["part_c", "part_m", "part_r"][k]].values
    ax.barh(r.index, v, left=gauche, color=col, height=.62, label=lab, edgecolor=SURFACE, lw=2)
    for i, (g, w) in enumerate(zip(gauche, v)):
        if w > .07: ax.text(g + w/2, i, f"{w*100:.0f} %", ha="center", va="center", fontsize=9,
                            color="white" if lab != "Revenus" else INK)
    gauche += v
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", colors=INK)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(.5, -.09), fontsize=9.5)
titre(ax, "La jambe revenus porte l'essentiel de la valeur retenue",
      "Part de chaque approche dans la valeur hybride, après pondération par la confiance")
plt.show()

## 6 · Correction : trier sans gonfler

Avant : un facteur qualité saisi à la main, une décote fixe. Après : le score de conformité pilote les trois approches. Résultat : les jeux bien documentés gagnent, les autres perdent, et le portefeuille bouge à peine.

In [ ]:
r = R.sort_values("ecart")
fig, ax = plt.subplots(figsize=(11, 4.8))
for i, (c, row) in enumerate(r.iterrows()):
    col = UP if row.ecart >= 0 else DOWN
    ax.plot([row.avant/1e6, row.apres/1e6], [i, i], color=col, lw=2.2, alpha=.55, solid_capstyle="round")
    ax.scatter(row.avant/1e6, i, s=34, color="white", edgecolor=MUTED, lw=1.2, zorder=3)
    ax.scatter(row.apres/1e6, i, s=60, color=col, zorder=4)
    ax.text(max(row.avant, row.apres)/1e6 + .07, i, pct(row.ecart), va="center", fontsize=9.5, color=INK2)
ax.set_yticks(range(len(r)), r.index, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel("valeur hybride, millions d'euros (avant attribution)")
ax.scatter([], [], s=34, color="white", edgecolor=MUTED, label="Avant (facteur saisi)")
ax.scatter([], [], s=60, color=UP, label="Après (piloté par le score)")
ax.legend(loc="lower right", fontsize=9.5)
titre(ax, f"Chaque jeu bouge de {pct(R.ecart.min(),0)} à {pct(R.ecart.max(),0)}, le portefeuille de {pct(TOT_AP/TOT_AV-1)}",
      "Bleu : le jeu gagne de la valeur · rouge : il en perd")
plt.show()

## 7 · Attribution : ce qui revient réellement

Quatre jeux sont détenus en coentreprise. La quote-part réelle n'est pas connue : l'étude retient 50 % par hypothèse prudente, et ne la devine pas plus finement. Sans cette correction, les actifs en coentreprise porteraient l'essentiel de la valeur affichée.

In [ ]:
brut = R.apres.sum()
pertes = (R.apres - R.attrib)[R.part < 1].sort_values(ascending=False)
etapes = [("Valeur hybride\n100 %", brut, "total")] + [(f"{c}\n−50 %", -v, "delta") for c, v in pertes.items()] + [("Valeur\nattribuable", TOT_AP, "total")]
fig, ax = plt.subplots(figsize=(11, 4.6))
cumul = 0
for i, (lab, v, t) in enumerate(etapes):
    if t == "total":
        ax.bar(i, v/1e6, color=BLUE, width=.6); cumul = v
        ax.text(i, v/1e6 + .12, eur(v), ha="center", fontsize=10, fontweight="semibold")
    else:
        ax.bar(i, -v/1e6, bottom=(cumul + v)/1e6, color=DOWN, alpha=.85, width=.6)
        ax.text(i, cumul/1e6 + .12, eur(v), ha="center", fontsize=9, color=INK2)
        cumul += v
ax.set_xticks(range(len(etapes)), [e[0] for e in etapes], fontsize=9.5, color=INK)
ax.grid(axis="x", visible=False); ax.set_ylabel("millions d'euros")
part_jv = R.apres[R.part < 1].sum() / brut
titre(ax, f"La quote-part en coentreprise retire {(1 - TOT_AP/brut)*100:.0f} % de la valeur brute",
      f"Les actifs en coentreprise portent {part_jv*100:.0f} % de la valeur brute · quote-part retenue : 50 % (hypothèse)")
plt.show()

## 8 · Exposition réglementaire et priorités

Cinq critères notés de 1 à 5 mesurent le risque qu'un jeu soit un jour classé « donnée importante », donc soumis à des restrictions de transfert. Le critère dominant (30 %) est la **maturité de l'industrie locale** : plus elle rattrape, plus le régulateur protège la donnée du secteur.

La priorité croise la valeur (normalisée sur 100) et l'exposition, avec deux seuils à 50 :

| | Exposition < 50 | Exposition ≥ 50 |
|---|---|---|
| **Valeur ≥ 50** | P2 · valoriser sans urgence | **P1 · valoriser en priorité** |
| **Valeur < 50** | P4 · différer | P3 · surveiller et sécuriser |

In [ ]:
w = np.array(list(EXPO_POIDS.values()))
R["expo"] = (EXPO.values @ w) / 5 * 100
R["valeur_norm"] = R.apres / R.apres.max() * 100
def prio(v, e):
    if v >= P["seuilValeur"]: return "P1" if e >= P["seuilExpo"] else "P2"
    return "P3" if e >= P["seuilExpo"] else "P4"
R["priorite"] = [prio(v, e) for v, e in zip(R.valeur_norm, R.expo)]
LIB = {"P1": "Valoriser en priorité", "P2": "Valoriser sans urgence", "P3": "Surveiller et sécuriser", "P4": "Différer"}

fig, ax = plt.subplots(figsize=(11, 6.4))
ax.axhspan(50, 105, xmin=0, xmax=1, color=GRID, alpha=.0)
ax.add_patch(plt.Rectangle((50, 50), 55, 55, color=PRIO["P1"], alpha=.07, lw=0))
ax.axvline(P["seuilValeur"], color=AXIS, ls="--", lw=1); ax.axhline(P["seuilExpo"], color=AXIS, ls="--", lw=1)
for q, (x, y) in {"P1": (101, 101), "P2": (101, 4), "P3": (2, 101), "P4": (2, 4)}.items():
    ax.text(x, y, f"{q} · {LIB[q]}", ha="right" if x > 50 else "left", va="top" if y > 50 else "bottom",
            fontsize=9.5, color=PRIO[q], fontweight="semibold")
decal = {"Énergie": (8, -12), "Personnel": (8, 6), "Télémétrie": (8, -12), "Appro": (8, 8),
         "Fournisseurs": (8, -12), "Maintenance": (-12, 10), "Procédés": (12, -16), "Essais": (-8, 10)}
for c, row in R.iterrows():
    taille = 60 + row.attrib / R.attrib.max() * 520
    ax.scatter(row.valeur_norm, row.expo, s=taille, color=PRIO[row.priorite], alpha=.85, edgecolor="white", lw=2, zorder=3)
    dx, dy = decal[c]
    ax.annotate(c, (row.valeur_norm, row.expo), xytext=(dx, dy), textcoords="offset points",
                ha="left" if dx > 0 else "right", fontsize=10, color=INK)
ax.set_xlim(0, 104); ax.set_ylim(0, 104)
ax.set_xlabel("valeur normalisée (100 = jeu le plus valorisé)"); ax.set_ylabel("score d'exposition (0 à 100)")
titre(ax, "Trois jeux cumulent forte valeur et forte exposition",
      "Taille du cercle : valeur attribuable · quadrant ombré : priorité P1")
plt.show()

recap = R[["verdict","attrib","expo","valeur_norm","priorite"]].copy().sort_values(["priorite","attrib"], ascending=[True, False])
recap.insert(0, "Jeu", recap.index); recap["priorite"] = recap.priorite + " · " + recap.priorite.map(LIB)
recap.columns = ["Jeu","Verdict","Valeur attribuable","Exposition","Valeur normalisée","Priorité"]
tableau(recap, {"Valeur attribuable": eur, "Exposition": "{:.0f}", "Valeur normalisée": "{:.0f}"}, barre=["Exposition"])

## 9 · Gains par critère : où investir l'effort

Pour chaque jeu, ce que rapporterait le passage d'un critère à 5, à hypothèses inchangées. La sensibilité de la valeur au score est calculée numériquement, puis répartie selon le poids de chaque critère. Une case vide signifie que le critère est déjà au maximum.

In [ ]:
NB = CRITERES.bloc.value_counts()
POIDS_BLOC = {"A": P["wDoc"], "B": P["wQual"], "C": P["wApp"]}
G = pd.DataFrame(index=CRITERES.code, columns=R.index, dtype=float)
for _, j in JEUX.iterrows():
    s = R.score[j.court]
    sens = valoriser(j, s + 1)["val"] - valoriser(j, s)["val"]          # € par point de score
    for _, c in CRITERES.iterrows():
        G.loc[c.code, j.court] = (5 - NOTES.loc[c.code, j.court]) * POIDS_BLOC[c.bloc] * 25 / NB[c.bloc] * sens
G = G[R.score.sort_values().index]
fig, ax = plt.subplots(figsize=(11.5, 7.2))
Gm = np.ma.masked_where(G.values <= 0, G.values / 1e3)
ax.imshow(Gm, cmap=mpl.colors.LinearSegmentedColormap.from_list("b", ["#e9f1fb", "#6da7ec", "#0d366b"]), aspect="auto")
for (i, k), v in np.ndenumerate(G.values):
    if v > 0: ax.text(k, i, f"{v/1e3:.0f}", ha="center", va="center", fontsize=9, color="white" if v/1e3 > G.values.max()/1e3*.45 else INK)
ax.set_xticks(range(G.shape[1]), G.columns, color=INK); ax.xaxis.tick_top()
ax.set_yticks(range(17), [f"{c}  {n}" for c, n in zip(CRITERES.code, CRITERES["critère"])], color=INK2, fontsize=9.5)
ax.grid(False); ax.tick_params(length=0)
for sp in ax.spines.values(): sp.set_visible(False)
for y in (3.5, 11.5): ax.axhline(y, color=SURFACE, lw=4)
titre_fig(fig, ax, "Ce que rapporte chaque critère porté à 5, en milliers d'euros",
          "Colonnes du jeu le moins conforme au plus conforme · valeur du jeu, avant attribution · case vide : déjà à 5")
plt.show()

top = G.stack().sort_values(ascending=False).head(8).reset_index()
top.columns = ["Critère", "Jeu", "Gain"]
top["Critère"] = top["Critère"] + " · " + top["Critère"].map(dict(zip(CRITERES.code, CRITERES["critère"])))
display(Markdown("**Les huit chantiers les plus rentables du portefeuille**"))
tableau(top[["Jeu", "Critère", "Gain"]], {"Gain": eur}, barre=["Gain"])

## 10 · Sensibilité : quelles hypothèses comptent

Chaque barre montre la valeur attribuable du portefeuille quand une seule hypothèse passe à sa borne basse puis haute, toutes les autres restant fixes. Les barres les plus longues désignent les hypothèses à défendre en priorité devant un interlocuteur sceptique.

In [ ]:
def portefeuille(Pm=None, mult=None):
    Px = {**P, **(Pm or {})}; J = JEUX.copy()
    for col, m in (mult or {}).items(): J[col] = J[col] * m
    return calculer(Px, J).attrib.sum()

base = portefeuille()
tests = [
    ("Taux d'actualisation (10 % / 15 %)", {"taux": .15}, None, {"taux": .10}, None),
    ("Bénéfices et économies (−30 % / +30 %)", None, {"ben": .7, "eco": .7}, None, {"ben": 1.3, "eco": 1.3}),
    ("Confiance revenus (0,55 / 0,85)", {"cr": .55}, None, {"cr": .85}, None),
    ("Poids revenus (35 % / 55 %)", {"pr": .35, "pc": .35, "pm": .30}, None, {"pr": .55, "pc": .25, "pm": .20}, None),
    ("Prix de référence (−30 % / +30 %)", None, {"prix": .7}, None, {"prix": 1.3}),
    ("Coût de reconstitution (−20 % / +20 %)", None, {"etp": .8, "infra": .8}, None, {"etp": 1.2, "infra": 1.2}),
    ("Effet du score sur les revenus β (0,25 / 0,75)", {"beta": .25}, None, {"beta": .75}, None),
    ("Effet du score sur la décote α (0,15 / 0,45)", {"alpha": .15}, None, {"alpha": .45}, None),
]
lignes = []
for lab, pb, mb, ph, mh in tests:
    lignes.append((lab, portefeuille(pb, mb) - base, portefeuille(ph, mh) - base))
S = pd.DataFrame(lignes, columns=["hyp", "bas", "haut"])
S["amp"] = (S.haut - S.bas).abs(); S = S.sort_values("amp")
fig, ax = plt.subplots(figsize=(11, 4.9))
for i, row in enumerate(S.itertuples()):
    for v in (row.bas, row.haut):
        ax.barh(i, v/1e6, color=UP if v >= 0 else DOWN, height=.58)
        ax.text(v/1e6 + (.03 if v >= 0 else -.03), i, pct(v/base, 0), va="center",
                ha="left" if v >= 0 else "right", fontsize=9, color=INK2)
ax.axvline(0, color=INK, lw=1)
m = max(S.bas.abs().max(), S.haut.abs().max()) / 1e6; ax.set_xlim(-m * 1.22, m * 1.22)
ax.set_yticks(range(len(S)), S.hyp, color=INK); ax.grid(axis="y", visible=False)
ax.set_xlabel(f"écart à la valeur centrale ({eur(base)}), millions d'euros")
titre(ax, "Les hypothèses de revenus dominent le résultat",
      "Une hypothèse à la fois · bleu : la valeur monte · rouge : elle baisse")
plt.show()

## 11 · Monte Carlo : la fourchette plutôt que le point

10 000 tirages. Chaque hypothèse incertaine suit une loi triangulaire autour de sa valeur centrale ; le score de conformité est tenu fixe pour isoler l'incertitude économique. Graine fixée : le résultat est reproductible.

| Hypothèse | Bas | Central | Haut |
|---|---|---|---|
| ETP-mois, infrastructure | ×0,80 | ×1 | ×1,25 |
| Coût d'un ETP | ×0,90 | ×1 | ×1,10 |
| Volume | ×0,80 | ×1 | ×1,20 |
| Prix de référence | ×0,60 | ×1 | ×1,50 |
| Décote | −10 pts | 0 | +10 pts |
| Bénéfices, économies | ×0,60 | ×1 | ×1,30 |
| Taux d'actualisation | 10 % | 12 % | 15 % |

In [ ]:
rng = np.random.default_rng(2026)
N = 10_000
tri = lambda a, m, b, n=N: rng.triangular(a, m, b, n)
sims = {}
for _, j in JEUX.iterrows():
    s = R.score[j.court]
    f   = borne(P["fMin"] + (s - 20) / 80 * (P["fMax"] - P["fMin"]), P["fMin"], P["fMax"])
    dec = borne(j.dec - P["alpha"]*(s - 50)/100 + tri(-.10, 0, .10), P["decMin"], P["decMax"])
    mul = 1 + P["beta"]*(s - 50)/100
    t   = tri(.10, .12, .15)
    vc = (j.etp*tri(.8,1,1.25)*j.cetp*tri(.9,1,1.1) + j.infra*tri(.8,1,1.25)) * f
    vm = j.vol*tri(.8,1,1.2) * j.prix*tri(.6,1,1.5) * (1 - dec)
    vr = (j.ben*tri(.6,1,1.3) + j.eco*tri(.6,1,1.3)) * mul * (1 - (1+t)**(-j.hor)) / t
    wc, wm, wr = P["pc"]*P["cc"], P["pm"]*j.confm, P["pr"]*P["cr"]
    sims[j.court] = (wc*vc + wm*vm + wr*vr) / (wc + wm + wr) * j.part
SIM = pd.DataFrame(sims); PF = SIM.sum(axis=1)
q = SIM.quantile([.05, .5, .95]).T.sort_values(.5)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
for i, (c, row) in enumerate(q.iterrows()):
    a1.plot([row[.05]/1e6, row[.95]/1e6], [i, i], color="#9ec5f4", lw=7, solid_capstyle="round")
    a1.scatter(row[.5]/1e6, i, s=55, color=BLUE, edgecolor="white", lw=1.5, zorder=3)
    a1.text(row[.95]/1e6 + .04, i, eur(row[.5]), va="center", fontsize=9, color=INK2)
a1.set_yticks(range(len(q)), q.index, color=INK); a1.grid(axis="y", visible=False)
a1.set_xlabel("valeur attribuable, millions d'euros")
titre(a1, "Fourchette à 90 % par jeu", "Barre : 5e au 95e centile · point : médiane")

a2.hist(PF/1e6, bins=60, color="#6da7ec", edgecolor=SURFACE, lw=.4)
for v, l in ((PF.quantile(.05), "P5"), (PF.median(), "médiane"), (PF.quantile(.95), "P95")):
    a2.axvline(v/1e6, color=INK if l == "médiane" else MUTED, lw=1.2, ls="-" if l == "médiane" else "--")
    a2.text(v/1e6, a2.get_ylim()[1]*.97, f"{l}\n{eur(v)}", ha="center", va="top", fontsize=9, color=INK,
            bbox=dict(fc=SURFACE, ec="none", pad=1.5))
a2.set_xlabel("valeur attribuable du portefeuille, millions d'euros"); a2.set_yticks([])
a2.spines["left"].set_visible(False); a2.grid(axis="y", visible=False)
titre(a2, "Le portefeuille sur 10 000 scénarios", f"Valeur centrale du modèle : {eur(TOT_AP)}")
plt.tight_layout(); plt.show()

print(f"Portefeuille : médiane {eur(PF.median())} · intervalle à 90 % [{eur(PF.quantile(.05))} ; {eur(PF.quantile(.95))}]"
      f" · largeur relative {((PF.quantile(.95)-PF.quantile(.05))/PF.median())*100:.0f} %")

**Lecture.** La médiane simulée est un peu sous la valeur centrale du modèle : les hypothèses de bénéfices sont volontairement asymétriques (jusqu'à −40 %, mais seulement +30 %), parce qu'un plan de bénéfices déçoit plus souvent qu'il ne surprend. Les fourchettes des jeux se chevauchent fortement entre jeux voisins : l'ordre des deux ou trois premiers est robuste, celui des jeux du milieu l'est beaucoup moins. C'est un argument de plus pour présenter un **classement par groupes** (P1 à P4) plutôt qu'un classement au rang près.

## 12 · Pondération : dériver les poids au lieu de les poser

Les poids 30 / 25 / 45 sont posés au jugement. Le cadre DATAMITE (Cork, Aix-la-Chapelle) les dérive par **comparaisons par paires** (ANP, dont l'AHP est le cas sans interdépendance). Le principe : on compare les approches deux à deux sur l'échelle de Saaty (1 = égale importance, 3 = modérément plus, 5 = nettement plus), le vecteur propre principal de la matrice donne les poids, et un **ratio de cohérence** sous 10 % garantit que les jugements ne se contredisent pas.

Deux principes orientent l'application :
- le niveau d'agrégation dépend de l'usage : **par jeu** si la donnée est vendue, **au niveau global** si elle sert un service interne. Ici l'usage est interne, donc une seule matrice pour tout le portefeuille ;
- au-delà d'une dizaine d'éléments comparés, les comparaisons peuvent être « adoucies » pour absorber les incohérences.

La matrice ci-dessous est **illustrative** : elle traduit les jugements implicites de l'étude et devra être remplie avec des experts métier.

In [ ]:
noms = ["Coût", "Marché", "Revenus"]
A = np.array([[1,   2,   1/2],     # le coût est plus fiable que le marché, moins pertinent que les revenus
              [1/2, 1,   1/3],
              [2,   3,   1  ]])
vp, vec = np.linalg.eig(A)
k = np.argmax(vp.real); w_ahp = np.abs(vec[:, k].real); w_ahp /= w_ahp.sum()
lam = vp.real[k]; n = len(A); CI = (lam - n) / (n - 1); RI = {3: .58, 4: .90, 5: 1.12}[n]; CR = CI / RI

display(Markdown(f"**Ratio de cohérence : {CR*100:.1f} %** ({'cohérent' if CR < .1 else 'à revoir'}, seuil 10 %) · λ max = {lam:.3f}".replace(".", ",")))
w_jug = np.array([P["pc"], P["pm"], P["pr"]])
v_ahp = calculer({**P, "pc": w_ahp[0], "pm": w_ahp[1], "pr": w_ahp[2]})

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.25]})
x = np.arange(3)
a1.bar(x - .19, w_jug, width=.36, color=MUTED, label="Jugement (étude)")
a1.bar(x + .19, w_ahp, width=.36, color=[BLUE, ORANGE, AQUA], label="Dérivés (AHP)")
for i in range(3):
    a1.text(i - .19, w_jug[i] + .01, f"{w_jug[i]*100:.0f} %", ha="center", fontsize=9, color=INK2)
    a1.text(i + .19, w_ahp[i] + .01, f"{w_ahp[i]*100:.0f} %", ha="center", fontsize=9, color=INK)
a1.set_xticks(x, noms, color=INK); a1.set_yticks([]); a1.spines["left"].set_visible(False); a1.grid(False)
a1.legend(fontsize=9, loc="upper left")
titre(a1, "Poids posés et poids dérivés", "Gris : jugement · couleur : comparaisons par paires")

comp = pd.DataFrame({"Jugement": R.attrib, "AHP": v_ahp.attrib}).sort_values("Jugement")
for i, (c, row) in enumerate(comp.iterrows()):
    a2.plot([row.Jugement/1e6, row.AHP/1e6], [i, i], color=GRID, lw=2)
    a2.scatter(row.Jugement/1e6, i, s=36, color=MUTED, zorder=3)
    a2.scatter(row.AHP/1e6, i, s=48, color=BLUE, zorder=4)
a2.set_yticks(range(len(comp)), comp.index, color=INK); a2.grid(axis="y", visible=False)
a2.set_xlabel("valeur attribuable, millions d'euros")
rang_j, rang_a = R.attrib.rank(ascending=False), v_ahp.attrib.rank(ascending=False)
bouge = [c for c in R.index if rang_j[c] != rang_a[c]]
prio_a = [prio(v, e) for v, e in zip(v_ahp.apres / v_ahp.apres.max() * 100, R.expo)]
prio_identiques = list(R.priorite) == prio_a
t2 = "Le classement ne bouge pas" if not bouge else f"{len(bouge)} jeux échangent leur rang, les priorités {'restent' if prio_identiques else 'changent'}"
titre(a2, t2, f"Portefeuille : {eur(R.attrib.sum())} → {eur(v_ahp.attrib.sum())}")
plt.tight_layout(); plt.show()

ecart_max = ((v_ahp.attrib / R.attrib) - 1).abs().max()
print("Jeux qui changent de rang :", ", ".join(f"{c} ({rang_j[c]:.0f}e → {rang_a[c]:.0f}e)" for c in bouge) or "aucun")
print("Catégories P1 à P4 identiques :", "oui" if prio_identiques else "non", "· écart maximal par jeu :", pct(ecart_max))

**Lecture.** Avec des jugements cohérents, les comparaisons par paires donnent plus de poids aux revenus que l'étude (54 % contre 45 %) et moins au marché. La valeur totale monte donc mécaniquement, ce qui rappelle qu'**un montant dépend des poids, pas seulement des données**. Le résultat qui compte est ailleurs : les catégories de priorité ne bougent pas, et seuls deux jeux de valeur très proche échangent leur rang. La décision tient, quelle que soit la méthode de pondération.

## 13 · Ancrage externe

### 13.1 Face au ticket moyen national

Statistiques publiques arrêtées en juin 2025 : 3 364 jeux listés sur les bourses de données pour environ 4 milliards de yuans de transactions cumulées, soit un **ticket moyen d'environ 154 k€** par jeu (au cours de 7,73 yuans pour un euro). Ce comparable agrège des données publiques et privées de toute nature : il sert d'ordre de grandeur, pas de prix.

In [ ]:
TICKET = 4e9 / 3364 / 7.73
mult = (R.attrib / TICKET).sort_values()
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.barh(mult.index, mult.values, color=[BLUE if m < 10 else "#0d366b" for m in mult], height=.6)
for i, m in enumerate(mult): ax.text(m + .15, i, f"×{m:.1f}".replace(".", ","), va="center", fontsize=9.5, color=INK2)
ax.axvline(1, color=INK, lw=1); ax.text(1.1, len(mult) - .45, "ticket moyen national", fontsize=9, color=INK2)
ax.grid(axis="y", visible=False); ax.tick_params(axis="y", colors=INK)
ax.set_xlabel("multiple du ticket moyen national (valeur attribuable / 154 k€)")
titre(ax, f"Le portefeuille vaut en moyenne ×{(R.attrib.mean()/TICKET):.1f} le ticket national".replace(".", ","),
      "Un écart à énoncer, pas à cacher : jeux industriels spécialisés contre moyenne toutes catégories")
plt.show()

### 13.2 L'écart entre valeur perçue et valeur reconnue

L'enquête de Gao, Liu et Lu (version du 13 septembre 2026) auprès des sociétés cotées chinoises mesure ce que les entreprises *pensent* que valent leurs données, et ce qu'elles inscrivent *réellement* au bilan depuis que la règle le permet (janvier 2024).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
lab = ["Valeur estimée\nau coût historique", "Valeur estimée\nen valeur économique", "Montant réellement\ninscrit au bilan"]
val = [12.15, 17.15, 0.26]
ax.bar(lab, val, color=["#6da7ec", BLUE, CRIT], width=.5)
for i, v in enumerate(val): ax.text(i, v + .4, f"{v:.2f} %".replace(".", ","), ha="center", fontsize=12, fontweight="semibold")
ax.set_ylabel("part de l'actif total"); ax.set_ylim(0, 20); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", colors=INK)
titre(ax, "La valeur de la donnée vit presque entièrement hors du bilan",
      "Sociétés cotées chinoises · 132 sur plus de 5 000 (2,6 %) ont inscrit des données fin 2025 · Gao, Liu et Lu (2026)")
plt.show()

**Ce que ça change pour l'étude.** La voie comptable est étroite : la règle est prospective (les dépenses antérieures à 2024 sont perdues pour le bilan), et la reconnaissance bute sur l'évaluation et les droits. La grille se place donc sur le terrain de la **valorisation managériale** : classer, prioriser, décider quoi documenter d'abord. C'est là que la valeur est mobilisable aujourd'hui.

Un point de repère complémentaire :
- au 30 avril 2026, **136 sociétés A-share** avaient inscrit **3,79 milliards de yuans** de données, dont 82 % en immobilisations incorporelles ; les trois opérateurs télécoms en portent 55 %, et l'industrie manufacturière compte le plus grand nombre de sociétés (32).

## 14 · Extension : la valeur des flux futurs d'un parc installé

Le portefeuille actuel ne dit rien des données que produiront les équipements livrés demain. Ordre de grandeur, avec des hypothèses **purement illustratives**, volontairement simples et toutes réglables : une valeur captable par unité et par an, une durée de vie, un taux de capture réaliste.

In [ ]:
H = dict(valeur_unite_an=20_000, duree=15, taux=.12, capture=.35, commande=10, parc=1_000)   # paramètres illustratifs
fa = annuite(H["duree"], H["taux"])
par_unite = H["valeur_unite_an"] * fa * H["capture"]
cartes = [("par unité livrée", eur(par_unite), "valeur actualisée captable sur la durée de vie"),
          (f"pour une commande de {H['commande']} unités", eur(par_unite * H["commande"]), "la couche de données cachée derrière la vente"),
          ("flux annuel du parc installé", eur(H["valeur_unite_an"] * H["capture"] * H["parc"]), f"{H['parc']:,}".replace(",", " ") + " unités en service, au taux de capture")]
fig = plt.figure(figsize=(12, 2.6))
for i, (l, v, s) in enumerate(cartes):
    ax = fig.add_axes([.005 + i*.333, .05, .32, .88]); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=.06", transform=ax.transAxes, fc=CARD, ec="none"))
    ax.text(.07, .64, v, fontsize=24, fontweight="semibold", color=BLUE, transform=ax.transAxes)
    ax.text(.07, .36, l, fontsize=11.5, transform=ax.transAxes)
    ax.text(.07, .16, s, fontsize=9.5, color=MUTED, transform=ax.transAxes)
plt.show()

cap = np.linspace(.1, .6, 51)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(cap*100, H["valeur_unite_an"] * fa * cap * H["commande"] / 1e6, color=BLUE, lw=2)
ax.axvline(H["capture"]*100, color=MUTED, ls="--", lw=1)
ax.text(H["capture"]*100 + .6, ax.get_ylim()[1]*.9 if ax.get_ylim()[1] else 1, f"hypothèse retenue : {H['capture']*100:.0f} %", fontsize=9, color=INK2)
ax.set_xlabel("taux de capture (%)"); ax.set_ylabel("millions d'euros")
titre(ax, "La couche de données d'une commande dépend d'abord du taux de capture",
      f"Commande de {H['commande']} unités · {eur(H['valeur_unite_an'])} par unité et par an · {H['duree']} ans à {H['taux']*100:.0f} %")
plt.show()

> Cette valeur future reste **hors grille** : elle dépend d'hypothèses commerciales, pas de la qualité des jeux existants. Elle sert à montrer l'ordre de grandeur de l'enjeu, pas à le chiffrer pour une décision.

## 15 · Veille réglementaire : le calendrier jusqu'en octobre 2026

La séquence compte plus que chaque texte : l'État a **recensé avant de tarifer**, et chaque étape a suivi la précédente à quelques mois d'écart. Tous les dispositifs publics de tarification portent sur les **données publiques** ; les données d'entreprise relèvent de la négociation libre.

In [ ]:
EV = [
 ("2020-04", "La donnée devient facteur de production", "Socle doctrinal", MUTED),
 ("2022-12", "« Vingt articles » : droits séparés sur la donnée", "Socle doctrinal", MUTED),
 ("2023-08", "Règle comptable : la donnée peut entrer au bilan", "Comptabilité", BLUE),
 ("2023-10", "Création de l'Administration nationale des données", "Institutions", MUTED),
 ("2024-05", "Première liste négative d'export de données (zone franche)", "Transferts", ORANGE),
 ("2024-08", "Liste négative municipale, secteurs stratégiques inclus", "Transferts", ORANGE),
 ("2024-10", "Texte-cadre « 1 » sur les données publiques", "Données publiques", AQUA),
 ("2025-03", "Entrée en vigueur des trois textes « 3 » (registre, licence, prix)", "Données publiques", AQUA),
 ("2025-08", "Lignes directrices : jeux de données de haute qualité (17 critères)", "Qualité", BLUE),
 ("2026-03", "Action « fondations des données industrielles », horizon fin 2026", "Industrie", BLUE),
 ("2026-06", "Suspension des nouvelles titrisations adossées à des données", "Finance", CRIT),
 ("2026-07", "Guide d'enregistrement des droits de propriété (essai)", "Droits", BLUE),
 ("2026-08", "Registre national : plus de 300 000 entrées · rapport du Congrès américain", "Données publiques", AQUA),
 ("2026-09", "Normes nationales de classification et d'enregistrement des actifs de données (1er sept.)", "Comptabilité", BLUE),
 ("2026-09", "Système national d'enregistrement des droits en ligne : 56 certificats le 1er jour", "Droits", BLUE),
 ("2026-10", "Finale nationale du concours « Data Element × » (20 au 23 oct.)", "Écosystème", MUTED),
]
ev = pd.DataFrame(EV, columns=["date", "évènement", "thème", "c"])
ev["d"] = pd.to_datetime(ev.date)
fig, ax = plt.subplots(figsize=(12, 8.4)); ax.axis("off")
n = len(ev); ys = np.arange(n)[::-1]
ax.plot([0.12, 0.12], [-.5, n - .5], color=AXIS, lw=1.2)
annee_prec = None
for y, row in zip(ys, ev.itertuples()):
    ax.scatter(.12, y, s=70, color=row.c, edgecolor="white", lw=1.6, zorder=3)
    an = row.date[:4]
    ax.text(.0, y, row.d.strftime("%m.%Y"), va="center", fontsize=9.5, color=INK if an != annee_prec else MUTED)
    annee_prec = an
    ax.text(.15, y, row.évènement, va="center", fontsize=10.5, color=INK)
    ax.text(.98, y, row.thème, va="center", ha="right", fontsize=9, color=MUTED)
ax.set_xlim(0, 1); ax.set_ylim(-.8, n)
ax.text(0, n + .1, "Du principe à l'infrastructure : six ans de construction", fontsize=14, fontweight="semibold")
plt.show()

**Les deux nouveautés de septembre et octobre 2026**

1. **Un système national d'enregistrement des droits de propriété** est en ligne depuis le 11 septembre, en phase d'essai : 56 certificats délivrés le premier jour, valables cinq ans, déposables via trois bourses de données nationales, et valables partout dans le pays. C'est la brique qui manquait entre la qualité (la grille) et l'actifisation.
2. **Côté normes comptables internationales**, le projet de l'IASB sur les actifs incorporels doit fixer sa direction au second semestre 2026. Les données n'y sont pas nommées à ce stade.

> **Réserve de périmètre.** Ni le registre public, ni l'exploitation sous licence, ni la tarification administrée ne s'appliquent directement aux données d'un acteur privé. Ils fournissent une **couche de benchmark**, pas une méthode d'évaluation.

## 16 · Convergence avec l'état de l'art

Trois travaux récents décrivent l'architecture que la grille a reconstruite pour un cas anonymisé.

In [ ]:
L = pd.DataFrame([
 ("Familles de méthodes", "Coût, marché, revenus", "Coût, marché, revenus (408 articles)", "Coûts, demande, prix dans Val", "Coût imposé à l'entrée au bilan"),
 ("Structure", "Conformité → valeur → exposition", "Appel à des cadres hybrides", "Screening → métriques → calcul", "Enquête + données d'archives"),
 ("Dimensions cœur", "Documentation, qualité, application", "Qualité comme composante critique", "Quality, Relevance, Utility", "Évaluation et droits comme freins"),
 ("Pondération", "Jugement (30/25/45)", "Non spécifiée", "ANP (comparaisons par paires)", "Sans objet"),
 ("Nature de la valeur", "Relative, pour classer", "Pas de méthode standard", "Valeur, pas un prix", "Perçue ≫ reconnue"),
 ("Validation empirique", "Cas anonymisé, 8 jeux", "Manquante (constat)", "Manquante (constat)", "132 sociétés cotées"),
], columns=["Composant", "La grille", "Rennes · Mohan et al. 2026", "Cork · Vyhmeister et al. déc. 2025", "Gao, Liu et Lu 2026"])
tableau(L)

**Ce que la convergence prouve, et ce qu'elle ne prouve pas.** Deux équipes indépendantes décrivent la même architecture que la grille : c'est un argument de **légitimité méthodologique**. Ce n'est pas une preuve que les montants sont justes. La contribution de ce travail est précisément là : instancier ce cadre sur un cas anonymisé, ce que les deux revues identifient comme le manque principal du champ.

## 17 · Limites assumées et prochaines étapes

| Limite | Ce qu'elle implique | Réponse prévue |
|---|---|---|
| Valeurs relatives, pas des prix | Ne jamais annoncer « ce jeu vaut X » | Benchmark externe pour toute fourchette de prix |
| Jambe revenus dominante | Le résultat dépend des hypothèses de bénéfices | Sensibilité (partie 10) et Monte Carlo (partie 11) |
| Poids posés au jugement | Objection « pourquoi 45 % ? » | Matrice AHP remplie par des experts métier |
| Quote-part en coentreprise inconnue | 50 % retenu par prudence | À remplacer par la quote-part réelle |
| Exposition notée à dire d'expert | Score indicatif, pas réglementaire | À confronter aux catalogues de données importantes du secteur |
| Étape de productisation non couverte | La grille saute du recensement à la valeur | Une brique de productisation (canvas de valorisation) à construire |
| Actifisation non mesurée | Un score élevé ≠ un actif | Enregistrement des droits et traitement comptable à étudier |
| Normes de la juridiction étudiée ≠ IFRS | Une inscription selon les normes locales ne passe pas telle quelle en consolidation | Vérifier le traitement en consolidation IFRS |

## Export

La cellule suivante enregistre les résultats dans `outputs/resultats_valorisation.csv`, pour les reprendre dans un tableur.

In [ ]:
sortie = R[["nom","entité","part","score","verdict","vc","vm","vr","avant","apres","ecart","attrib","expo","priorite"]].copy()
sortie.columns = ["jeu","detention","part","score","verdict","valeur_cout","valeur_marche","valeur_revenus",
                  "hybride_avant","hybride_apres","ecart","attribuable","exposition","priorite"]
import os
os.makedirs("../outputs", exist_ok=True)
sortie.round(4).to_csv("../outputs/resultats_valorisation.csv", sep=";", decimal=",", encoding="utf-8-sig")
print("Écrit : outputs/resultats_valorisation.csv")

## Sources

**Académiques**
- Mohan, S. K., Bharathy, G. et Jalan, A. (2026). *Enterprise Data Valuation: A Targeted Literature Review*. Journal of Economic Surveys, 40(1), 73-92. DOI 10.1111/joes.12705.
- Vyhmeister, E., Pietropaoli, B., Bub, U., Schneider, R. et Visentin, A. (déc. 2025). Cadre de valorisation des données du projet Horizon Europe DATAMITE (101092989). University College Cork, FIR e.V. RWTH Aachen. arXiv:2512.07664. https://arxiv.org/abs/2512.07664
- Gao, X., Liu, J. et Lu, H. (2026). *Accounting for Data Assets*. SSRN 5164215, version du 13 septembre 2026.
- Saaty, T. L. (2006). *The Analytic Network Process*. Springer.

**Institutionnelles et presse**
- U.S.-China Economic and Security Review Commission (18 août 2026). *The People's Republic of Data: How China Is Turning Data into Capital*. https://www.uscc.gov/research/peoples-republic-data-how-china-turning-data-capital
- Administration nationale des données, conférence de presse du 25 août 2026 (registre national, 300 000+ entrées).
- Xinhua (12 septembre 2026). Mise en ligne du système national d'enregistrement des droits de propriété des données. https://www.news.cn/fortune/20260912/63c960fce4cd4d6bbeb41f0d542d6370/c.html
- Securities Times (24 juin 2026). 136 sociétés A-share, 3,786 milliards de yuans de données inscrites. https://stcn.com/article/detail/3976954.html
- Administration nationale des données (9 octobre 2026). Finale du concours « Data Element × ». https://www.nda.gov.cn/sjj/zwgk/tzgg/1009/20261009165059083009494_pc.html
- IFRS Foundation (avril 2026). Point d'avancement du projet *Intangible Assets*. https://www.ifrs.org/content/dam/ifrs/meetings/2026/april/ac/ac4-iasb-update.pdf
- China Briefing (2025). *The "Negative Lists" Easing Data Export in China's FTZs*. https://www.china-briefing.com/news/china-data-export-policy-ftzs/

## Reproducibility & License

- **Exécution :** *Run All* suffit ; aucune donnée externe n'est lue. Python 3.13, `numpy`, `pandas`, `matplotlib` (versions dans `requirements.txt`).
- **Graine :** le Monte Carlo (partie 11) utilise `np.random.default_rng(2026)` ; les résultats sont identiques d'une exécution à l'autre.
- **Nature des chiffres :** hypothèses de travail cohérentes entre elles, sur un cas anonymisé ; classement relatif, pas un prix opposable.
- **Licence :** code sous licence MIT ; texte, figures et données sous CC BY 4.0.
- **Liens :** [dépôt GitHub](https://github.com/Kukai-7/industrial-data-valuation) · [jeu de données synthétique sur Hugging Face](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · [démonstration](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)